In [1]:
import pandas as pd
### Only for lake-wise data, grid data is already minimal of size

lake_general_properties_columns = ['Hylak_id']

# Fig.1 plotting data
# No data is required for this figure, this is a illustration figure made in Adobe Illustrator
fig_1_data_columns = []

# Fig. 2 plotting data
fig_2_data_columns = ['mean_area', 'seasonality_dominance_percentage']

# Fig. 3 plotting data
fig_3_data_columns = ['linear_trend_of_standard_deviation_percentage_per_period', 'Lake_type', 'ari_ix_uav', 'EXTENT']

# Fig. 4 plotting data
# Data for this fig requires all area columns, please save the all calculated data in the next cell in a separate file
fig_4_data_columns = []

# Fig. 5 plotting data
fig_5_data_columns = [
    'percentage_of_extreme_low_water_compared_with_long_term_changes', 
    'linear_trend_of_stl_trend_percentage_per_period',
    'linear_trend_of_standard_deviation_per_period',
    'percentage_of_extreme_low_water_compared_with_long_term_trends_plus_average_seasonlity',
    'min_deviations_of_low_water_extremes_from_annual_means_second_period',
    ]

# ED Fig. 1 plotting data
# Data for this figure is not uploaded to Code Ocean due to quota limit
ED_fig_1_data_columns = []

# ED Fig. 2 plotting data
# Data for this figure is not uploaded to Code Ocean due to quota limit
ED_fig_2_data_columns = []

# ED Fig. 3 plotting data
# Data for this figure is basin-wise 'hydrobasins_lev02_with_accuracy_metrics.pkl', training_records.csv, and hybas_lev02_v1c_merged_lake_count_added_gt100.geojson
ED_fig_3_data_columns = []

# ED Fig. 4 plotting data
ED_fig_4_data_columns = ['Pour_lat', 'total_variation_relative_to_total_area', 'mean_area']

# ED Fig. 5 plotting data
ED_fig_5_data_columns = ['linear_trend_of_standard_deviation_percentage_per_period', 'ppd_pk_sav', 'seasonality_dominance_percentage']

# ED Fig. 6 plotting data
ED_fig_6_data_columns = [
    'percentage_of_extreme_high_water_compared_with_long_term_trends_plus_average_seasonlity',
    'linear_trend_of_stl_trend_percentage_per_period',
    'linear_trend_of_standard_deviation_per_period',
    'max_deviations_of_high_water_extremes_from_annual_means_second_period'
    ]

# ED Fig. 7 plotting data
# Data for this figure only need grid data
ED_fig_7_data_columns = []

# ED Fig. 8 plotting data
# This figure is flowchart, no data is required
ED_fig_8_data_columns = []

# ED Fig. 9 plotting data
# Data for this figure is comparison with Xingdong
ED_fig_9_data_columns = []

# ED Fig. 10 plotting data
# Data for this figure is hydrobasins_lev02_with_statistics_with_missing_data_count_and_area_full_quarter.pkl
ED_fig_10_data_columns = []

all_columns = list(set(
    lake_general_properties_columns +
    fig_1_data_columns +
    fig_2_data_columns +
    fig_3_data_columns +
    fig_4_data_columns +
    fig_5_data_columns +
    ED_fig_1_data_columns +
    ED_fig_2_data_columns +
    ED_fig_3_data_columns +
    ED_fig_4_data_columns +
    ED_fig_5_data_columns +
    ED_fig_6_data_columns +
    ED_fig_7_data_columns +
    ED_fig_8_data_columns +
    ED_fig_9_data_columns +
    ED_fig_10_data_columns
))

lake_lse_gdf_path = '/WORK/Data/global_lake_area/area_csvs/lakes/csv/monthly_lake_surface_extent_with_all_analysis.csv'
lake_lse_gdf = pd.read_csv(lake_lse_gdf_path)

lake_lse_gdf[all_columns].to_pickle('/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lake_lse_gdf_for_figure_making.pkl')

/tmp/ipykernel_36163/3740618081.py:91: DtypeWarning: Columns (562,919) have mixed types. Specify dtype option on import or set low_memory=False.
  lake_lse_gdf = pd.read_csv(lake_lse_gdf_path)


In [2]:
# Save data for Fig. 4

from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import pickle

area_start_date = '2001-01-01'
area_end_date = '2024-01-01'
date_fmt = '%Y-%m-%d'
area_start_date = datetime.strptime(area_start_date, date_fmt)
area_end_date = datetime.strptime(area_end_date, date_fmt)
current_date = area_start_date
area_columns = []
while current_date < area_end_date:
    area_columns.append(current_date.strftime(date_fmt))
    current_date += relativedelta(months=1)

# process basin-wise data
basin_pkl_path = '/WORK/Data/global_lake_area/hydrobasins/merged/hydrobasins_lev03_with_statistics.pkl'
basin_gdf = pd.read_pickle(basin_pkl_path)
basin_gdf['linear_trend_of_stl_trend_per_period_sum'] = basin_gdf['linear_trend_of_stl_trend_per_period_sum'] * 10
basins_with_increasing_long_term_trends = basin_gdf[basin_gdf['linear_trend_of_stl_trend_per_period_sum'] > 0]
basins_with_decreasing_long_term_trends = basin_gdf[basin_gdf['linear_trend_of_stl_trend_per_period_sum'] < 0]
lake_lse_csv_path = '/WORK/Data/global_lake_area/area_csvs/lakes/csv/lakes_all_with_aridity_and_permafrost_type.csv'
lake_lse_gdf = pd.read_csv(lake_lse_csv_path)
lake_lse_gdf = lake_lse_gdf[area_columns + ['Lake_type'] + ['ari_ix_uav', 'EXTENT'] + ['Pour_lat', 'Pour_long']]

lake_lse_lat_column = 'Pour_lat'
lake_lse_lon_column = 'Pour_long'
lake_lse_gdf['geometry'] = [Point(xy) for xy in zip(lake_lse_gdf[lake_lse_lon_column], lake_lse_gdf[lake_lse_lat_column])]
lake_lse_gdf = gpd.GeoDataFrame(lake_lse_gdf, crs='EPSG:4326')
lakes_in_increasing_trend_basins = gpd.sjoin(lake_lse_gdf, basins_with_increasing_long_term_trends, how='inner', op='within')
lakes_in_decreasing_trend_basins = gpd.sjoin(lake_lse_gdf, basins_with_decreasing_long_term_trends, how='inner', op='within')

# data for figure a
natural_area_time_series = lake_lse_gdf[lake_lse_gdf['Lake_type'] == 1][area_columns].sum(axis=0) * 1e-6
reservoir_area_time_series = lake_lse_gdf[lake_lse_gdf['Lake_type'] != 1][area_columns].sum(axis=0) * 1e-6
natural_area_time_series_anomaly = natural_area_time_series - natural_area_time_series.mean()
reservoir_area_time_series_anomaly = reservoir_area_time_series - reservoir_area_time_series.mean()
total_area_time_series_anomaly = natural_area_time_series_anomaly + reservoir_area_time_series_anomaly
# data for figure b
increasing_natural_area_time_series = lakes_in_increasing_trend_basins[lakes_in_increasing_trend_basins['Lake_type'] == 1][area_columns].sum(axis=0) * 1e-6
increasing_reservoir_area_time_series = lakes_in_increasing_trend_basins[lakes_in_increasing_trend_basins['Lake_type'] != 1][area_columns].sum(axis=0) * 1e-6
increasing_natural_area_time_series_anomaly = increasing_natural_area_time_series - increasing_natural_area_time_series.mean()
increasing_reservoir_area_time_series_anomaly = increasing_reservoir_area_time_series - increasing_reservoir_area_time_series.mean()
increasing_total_area_time_series_anomaly = increasing_natural_area_time_series_anomaly + increasing_reservoir_area_time_series_anomaly
# data for figure c
decreasing_natural_area_time_series = lakes_in_decreasing_trend_basins[lakes_in_decreasing_trend_basins['Lake_type'] == 1][area_columns].sum(axis=0) * 1e-6
decreasing_reservoir_area_time_series = lakes_in_decreasing_trend_basins[lakes_in_decreasing_trend_basins['Lake_type'] != 1][area_columns].sum(axis=0) * 1e-6
decreasing_natural_area_time_series_anomaly = decreasing_natural_area_time_series - decreasing_natural_area_time_series.mean()
decreasing_reservoir_area_time_series_anomaly = decreasing_reservoir_area_time_series - decreasing_reservoir_area_time_series.mean()
decreasing_total_area_time_series_anomaly = decreasing_natural_area_time_series_anomaly + decreasing_reservoir_area_time_series_anomaly

# Save these as pkl
save_data = {}
save_data['natural_area_time_series'] = natural_area_time_series
save_data['reservoir_area_time_series'] = reservoir_area_time_series
save_data['natural_area_time_series_anomaly'] = natural_area_time_series_anomaly
save_data['reservoir_area_time_series_anomaly'] = reservoir_area_time_series_anomaly
save_data['total_area_time_series_anomaly'] = total_area_time_series_anomaly
save_data['increasing_natural_area_time_series'] = increasing_natural_area_time_series
save_data['increasing_reservoir_area_time_series'] = increasing_reservoir_area_time_series
save_data['increasing_natural_area_time_series_anomaly'] = increasing_natural_area_time_series_anomaly
save_data['increasing_reservoir_area_time_series_anomaly'] = increasing_reservoir_area_time_series_anomaly
save_data['increasing_total_area_time_series_anomaly'] = increasing_total_area_time_series_anomaly
save_data['decreasing_natural_area_time_series'] = decreasing_natural_area_time_series
save_data['decreasing_reservoir_area_time_series'] = decreasing_reservoir_area_time_series
save_data['decreasing_natural_area_time_series_anomaly'] = decreasing_natural_area_time_series_anomaly
save_data['decreasing_reservoir_area_time_series_anomaly'] = decreasing_reservoir_area_time_series_anomaly
save_data['decreasing_total_area_time_series_anomaly'] = decreasing_total_area_time_series_anomaly

# dump to pickle
with open('/WORK/Data/global_lake_area/data_for_codeocean_1230/fig2_time_series_data.pkl', 'wb') as f:
    pickle.dump(save_data, f)


/tmp/ipykernel_88251/2617460067.py:29: DtypeWarning: Columns (277) have mixed types. Specify dtype option on import or set low_memory=False.
  lake_lse_gdf = pd.read_csv(lake_lse_csv_path)
/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3448: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):
/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3448: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):
